# FingerBall Stage 2H — Manual-guided Hand Landmark Annotation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2H_Manual_Guided_Annotation.ipynb)

**目的：** リリース前後で背景やボールに重なって誤検出されたMediaPipe 21点を、元動画の座標系でドラッグ修正します。元データは保持し、修正点・遮蔽点・補間点の由来をCSVとJSONに保存します。

**主な操作：** フレームを選択 → 近くの点をドラッグ → 必要に応じて「遮蔽・不明」に設定 → キーフレームを保存 → ほかのフレームを編集。対象フレームを飛ばす・評価用フレームを確保することも可能です。

**重要な限界：** 修正するのは**2D投影座標だけ**です。MediaPipeの相対Zや3D関節角を「正しくなった」とは扱いません。遮蔽点を勝手に真値化しません。

## 0. セットアップ（Colab対応のMatplotlibウィジェット）

In [ ]:
%pip -q install "ipympl>=0.9.4" "ipywidgets>=8.0" "opencv-python-headless>=4.8" "pandas>=2" "matplotlib>=3.8"

In [ ]:
from google.colab import drive,output
drive.mount('/content/drive')
output.enable_custom_widget_manager()
%matplotlib widget
from pathlib import Path
import copy,json,datetime,zipfile
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display,clear_output
from matplotlib.lines import Line2D

## 1. 入力・出力の設定

元動画を使用してください。Stage 2Dの`release_zoom_175_210.mp4`は画像座標が変わっているため、このNotebookのアノテーションには使用できません。元動画の場所が見つからなければ`VIDEO_PATH`を明示指定します。

In [ ]:
PROJECT_PATH='/content/drive/MyDrive/母指回内トラッキング'
SOURCE_STAGE='stage2d'
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
CSV_PATH=''                       # 空ならStage 2Dの21点CSV
VIDEO_PATH=''                     # 空ならMyDrive/母指回内トラッキング配下から元動画を探索
FIRST_FRAME,LAST_FRAME=185,198
KEY_FRAMES=[187,189,190,191,193,196]
THROWING_HAND='Left'              # 左右は確認のうえ手動指定
CROP_SIZE=300                     # 元動画の画像ピクセル、表示の一辺
AUTO_SAVE=True
assert THROWING_HAND in ('Left','Right')
project=Path(PROJECT_PATH)
if not project.is_dir():raise FileNotFoundError(project)
if CSV_PATH:
    csv_path=Path(CSV_PATH)
else:
    base=project/f'FingerBall_v2_{SOURCE_STAGE}_results'
    csv_path=base/VIDEO_STEM/'hand_21_landmarks_long.csv'
    if not csv_path.is_file():
        hits=sorted(base.glob('*/hand_21_landmarks_long.csv'))
        if not hits:raise FileNotFoundError('CSV_PATHを指定してください。21点CSVがありません')
        csv_path=hits[0]
if not csv_path.is_file():raise FileNotFoundError(csv_path)
if VIDEO_PATH:
    video_path=Path(VIDEO_PATH)
else:
    candidates=[p for p in project.rglob('*') if p.is_file()
       and p.suffix.lower() in ('.mp4','.mov','.avi')
       and VIDEO_STEM in p.stem and 'zoom' not in p.stem.lower()
       and 'annotated' not in p.stem.lower()]
    candidates=sorted(candidates,key=lambda p:(p.name!=VIDEO_STEM+'.mp4',len(p.parts)))
    if not candidates:
        raise FileNotFoundError('元動画が見つかりません。VIDEO_PATHに元動画の絶対パスを指定してください')
    video_path=candidates[0]
if not video_path.is_file():raise FileNotFoundError(video_path)
if any(t in video_path.stem.lower() for t in ('zoom','annotated','comparison')):
    raise ValueError('座標の異なる加工動画は使用できません。元動画VIDEO_PATHを指定してください')
out_dir=project/'FingerBall_v2_stage2h_results'/csv_path.parent.name
out_dir.mkdir(parents=True,exist_ok=True)
state_file=out_dir/'manual_annotation_session.json'
print('CSV:',csv_path)
print('ORIGINAL video:',video_path)
print('Saving into:',out_dir)

## 2. 動画とLandmarksの読み込み・座標一致チェック

In [ ]:
source_df=pd.read_csv(csv_path)
required={'frame','landmark_id','x_px','y_px','z_relative','detected'}
if required-set(source_df.columns):
    raise ValueError('CSV必須列不足: '+str(required-set(source_df.columns)))
source_df['frame']=pd.to_numeric(source_df.frame,errors='raise').astype(int)
source_df['landmark_id']=pd.to_numeric(source_df.landmark_id,errors='raise').astype(int)
if source_df.duplicated(['frame','landmark_id']).any():
    raise ValueError('frame/landmark_idに重複があります')
for c in ['x_px','y_px','z_relative','detected']:
    source_df[c]=pd.to_numeric(source_df[c],errors='coerce')
cap=cv2.VideoCapture(str(video_path))
if not cap.isOpened():raise RuntimeError('元動画をOpenCVで開けません')
TOTAL=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS=float(cap.get(cv2.CAP_PROP_FPS))
if not (0<=FIRST_FRAME<=LAST_FRAME<TOTAL):raise ValueError('指定フレーム範囲が動画外です')
for norm,pix,extent in [('x_norm','x_px',W),('y_norm','y_px',H)]:
    if norm in source_df:
        good=source_df[[norm,pix]].dropna()
        good=good[good[norm].abs()>1e-4]
        if len(good)>10:
            est=float(np.median(good[pix]/good[norm]))
            if abs(est-extent)>3:
                cap.release()
                raise ValueError(f'CSVの画像寸法は約{est:.1f}ですが、指定動画は{extent}です。加工動画ではなく元動画を確認')
FRAME_IDS=list(range(FIRST_FRAME,LAST_FRAME+1))
def get_video_frame(f):
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
    ok,im=cap.read()
    if not ok:raise RuntimeError(f'frame {f} の読み込み失敗')
    return cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
original={};edits={};statuses={}
for f in FRAME_IDS:
    group=source_df[(source_df.frame==f)&source_df.landmark_id.between(0,20)].sort_values('landmark_id')
    arr=np.full((21,2),np.nan,float)
    if len(group)==21 and group.landmark_id.tolist()==list(range(21)):
        p=group[['x_px','y_px']].to_numpy(float)
        valid=np.isfinite(p).all(axis=1)&group.detected.fillna(0).to_numpy().astype(bool)
        arr[valid]=p[valid]
    original[f]=arr.copy()
    edits[f]=arr.copy()
    statuses[f]=['ai' if np.isfinite(arr[i]).all() else 'missing' for i in range(21)]
if state_file.is_file():
    saved=json.loads(state_file.read_text(encoding='utf-8'))
    if (saved.get('video_filename')==video_path.name and
        saved.get('csv_path')==str(csv_path) and saved.get('resolution')==[W,H]):
        for key,item in saved.get('frames',{}).items():
            f=int(key)
            if f not in edits:continue
            p=np.array(item['coords'],dtype=object)
            edits[f]=np.array([[np.nan if v is None else float(v) for v in pair] for pair in p],float)
            statuses[f]=list(item['statuses'])
        print('前回の修正セッションを復元しました')
    else:print('既存の保存ファイルは別動画/別CSVです。上書きしません（保存先を変更してください）')
print(f'Video: {W}x{H} / {TOTAL} frames / {FPS:.3f}fps')
print('Frame range:',FRAME_IDS[0],FRAME_IDS[-1])

## 3. ドラッグ修正GUI

画像内の点をマウスでドラッグしてください。**上側のツールバーのPan/Zoomを解除してから**操作します。

- プルダウンで対象フレームを変更。灰色の21点はAI推定、**橙の点は手動修正**、×印は遮蔽・不明です。
- 「遮蔽・不明にする」で選択点の座標を`NaN`として記録。見えない部分の位置を正解扱いしません。
- 「この点を画像クリックで配置」では、次のクリック位置に選択点を移動できます。検出が飛んでいる場合に使用してください。
- 「保存」で途中経過をGoogle Driveに保存。動画フレーム切替時にも自動保存できます。


In [ ]:
NAMES=['WRIST','THUMB_CMC','THUMB_MCP','THUMB_IP','THUMB_TIP',
       'INDEX_MCP','INDEX_PIP','INDEX_DIP','INDEX_TIP',
       'MIDDLE_MCP','MIDDLE_PIP','MIDDLE_DIP','MIDDLE_TIP',
       'RING_MCP','RING_PIP','RING_DIP','RING_TIP',
       'PINKY_MCP','PINKY_PIP','PINKY_DIP','PINKY_TIP']
FINGERS={'Thumb':[0,1,2,3,4],'Index':[0,5,6,7,8],
         'Middle':[0,9,10,11,12],'Ring':[0,13,14,15,16],
         'Little':[0,17,18,19,20]}
COLORS={'Thumb':'#f59e0b','Index':'#06b6d4','Middle':'#d946ef',
        'Ring':'#9ca3af','Little':'#64748b'}
def save_session():
    payload={'version':'stage2h-v1','timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
      'video_filename':video_path.name,'csv_path':str(csv_path),'resolution':[W,H],
      'hand':THROWING_HAND,'frames':{
        str(f):{'coords':[[None if not np.isfinite(x) else float(x) for x in row] for row in edits[f]],
                'statuses':statuses[f]} for f in FRAME_IDS}}
    tmp=state_file.with_suffix('.tmp')
    tmp.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
    tmp.replace(state_file)
def center_for_frame(f):
    a=edits[f]
    good=a[np.isfinite(a).all(axis=1)]
    if len(good):return np.median(good,axis=0)
    for near in sorted(FRAME_IDS,key=lambda x:abs(x-f)):
        good=edits[near][np.isfinite(edits[near]).all(axis=1)]
        if len(good):return np.median(good,axis=0)
    return np.array([W/2,H/2])
f_widget=widgets.SelectionSlider(options=FRAME_IDS,value=190 if 190 in FRAME_IDS else FRAME_IDS[0],
    description='Frame',continuous_update=False,layout=widgets.Layout(width='550px'))
pt_widget=widgets.Dropdown(options=[(f'{i:02d} {name}',i) for i,name in enumerate(NAMES)],
    value=8,description='Point',layout=widgets.Layout(width='260px'))
crop_widget=widgets.IntSlider(value=CROP_SIZE,min=120,max=700,step=20,
    description='Crop px',continuous_update=False,layout=widgets.Layout(width='340px'))
place_button=widgets.Button(description='次のクリックで点を配置',button_style='info')
missing_button=widgets.Button(description='遮蔽・不明にする',button_style='warning')
restore_button=widgets.Button(description='選択点をAI推定へ戻す')
undo_button=widgets.Button(description='直前の変更をUndo')
save_button=widgets.Button(description='セッション保存',button_style='success')
msg=widgets.HTML(value='スライダーでフレームを選び、点をドラッグしてください')
fig_anno,ax_anno=plt.subplots(figsize=(9,8))
fig_anno.canvas.header_visible=False
mode={'drag':None,'place':False,'history':[]}
def take_snapshot():
    f=f_widget.value
    mode['history'].append((f,edits[f].copy(),list(statuses[f])))
    mode['history']=mode['history'][-30:]
def repaint(*args):
    f=f_widget.value
    ax_anno.clear()
    im=get_video_frame(f)
    cx,cy=center_for_frame(f)
    half=crop_widget.value/2
    ax_anno.imshow(im,extent=(0,W,H,0))
    ax_anno.set_xlim(max(0,cx-half),min(W,cx+half))
    ax_anno.set_ylim(min(H,cy+half),max(0,cy-half))
    a=edits[f]
    for finger,idxs in FINGERS.items():
        for i,j in zip(idxs[:-1],idxs[1:]):
            if np.isfinite(a[[i,j]]).all():
                ax_anno.plot(a[[i,j],0],a[[i,j],1],
                             color=COLORS[finger],lw=2.1,alpha=.9)
    for i,pt in enumerate(a):
        if not np.isfinite(pt).all():continue
        status=statuses[f][i]
        color='#f59e0b' if status=='manual_visible' else ('#67e8f9' if status=='interpolated' else '#e5e7eb')
        ax_anno.scatter(pt[0],pt[1],s=65 if i==pt_widget.value else 20,
                        c=color,edgecolors='#0f172a',linewidths=.9,zorder=5)
        if i==pt_widget.value:
            ax_anno.annotate(f'{i} {NAMES[i]}',pt,xytext=(8,-12),
                             textcoords='offset points',color='yellow',fontsize=9)
    ax_anno.set_title(f'Frame {f} • manual hand: {THROWING_HAND} • point {pt_widget.value} • '
        f'{statuses[f][pt_widget.value]}',fontsize=12)
    ax_anno.set_xlabel('Original image X (pixels)')
    ax_anno.set_ylabel('Original image Y (pixels)')
    fig_anno.canvas.draw_idle()
def press(event):
    if event.inaxes!=ax_anno or event.xdata is None or event.ydata is None:return
    if fig_anno.canvas.toolbar_visible and getattr(fig_anno.canvas,'toolbar',None) is not None:
        if getattr(fig_anno.canvas.toolbar,'mode',''):return
    f=f_widget.value
    point=np.array([event.xdata,event.ydata],float)
    if mode['place']:
        i=pt_widget.value;take_snapshot()
        edits[f][i]=point;statuses[f][i]='manual_visible'
        mode['place']=False;place_button.button_style='info'
        msg.value=f'Frame {f}: {NAMES[i]} を配置しました'
        if AUTO_SAVE:save_session()
        repaint()
        return
    a=edits[f]
    dd=np.linalg.norm(a-point,axis=1)
    dd[~np.isfinite(dd)]=np.inf
    i=int(np.argmin(dd))
    if dd[i]>max(12,crop_widget.value*.07):
        msg.value='点から遠い位置です。プルダウン選択後「次のクリックで点を配置」を使ってください'
        return
    pt_widget.value=i
    take_snapshot()
    mode['drag']=(f,i)
def move(event):
    if mode['drag'] is None or event.xdata is None or event.ydata is None or event.inaxes!=ax_anno:return
    f,i=mode['drag']
    if f!=f_widget.value:return
    edits[f][i]=[np.clip(event.xdata,0,W-1),np.clip(event.ydata,0,H-1)]
    statuses[f][i]='manual_visible'
    repaint()
def release(event):
    if mode['drag'] is None:return
    f,i=mode['drag']
    mode['drag']=None
    if AUTO_SAVE:save_session()
    msg.value=f'Frame {f}: {NAMES[i]} 修正済み'
    repaint()
def mark_missing(_):
    f,i=f_widget.value,pt_widget.value
    take_snapshot();edits[f][i]=[np.nan,np.nan];statuses[f][i]='occluded'
    if AUTO_SAVE:save_session()
    msg.value=f'Frame {f} {NAMES[i]}: 遮蔽・不明'
    repaint()
def restore_point(_):
    f,i=f_widget.value,pt_widget.value
    take_snapshot();edits[f][i]=original[f][i]
    statuses[f][i]='ai' if np.isfinite(edits[f][i]).all() else 'missing'
    if AUTO_SAVE:save_session()
    repaint()
def undo(_):
    if not mode['history']:msg.value='Undo履歴がありません';return
    f,a,s=mode['history'].pop()
    edits[f]=a;statuses[f]=s;f_widget.value=f
    if AUTO_SAVE:save_session()
    repaint()
def enable_place(_):
    mode['place']=True;place_button.button_style='warning'
    msg.value=f'画像をクリックして {NAMES[pt_widget.value]} の位置を指定してください'
def on_frame_change(change):
    if change['name']=='value' and change['new']!=change['old']:
        mode['place']=False;place_button.button_style='info'
        mode['drag']=None;repaint()
def persist(_):
    save_session()
    msg.value=f'保存しました：{state_file}'
f_widget.observe(on_frame_change,names='value')
pt_widget.observe(repaint,names='value')
crop_widget.observe(repaint,names='value')
place_button.on_click(enable_place)
missing_button.on_click(mark_missing)
restore_button.on_click(restore_point)
undo_button.on_click(undo)
save_button.on_click(persist)
fig_anno.canvas.mpl_connect('button_press_event',press)
fig_anno.canvas.mpl_connect('motion_notify_event',move)
fig_anno.canvas.mpl_connect('button_release_event',release)
display(widgets.VBox([widgets.HBox([f_widget,pt_widget]),
  crop_widget,widgets.HBox([place_button,missing_button,restore_button]),
  widgets.HBox([undo_button,save_button]),msg]))
display(fig_anno.canvas)
repaint()

## 4. 編集品質の確認と保存

**手動修正（manual_visible）・遮蔽（occluded）・AI初期推定（ai）を区別**し、元のZ値は触りません。エクスポート後のCSVは2D座標の解析用であり、Stage 2Gの疑似3Dに無条件で投入しないでください。

In [ ]:
save_session()
audit=[]
for f in FRAME_IDS:
    for i in range(21):
        a=original[f][i];b=edits[f][i]
        shift=float(np.linalg.norm(a-b)) if np.isfinite(a).all() and np.isfinite(b).all() else np.nan
        audit.append({'frame':f,'landmark_id':i,'landmark_name':NAMES[i],
                      'status':statuses[f][i],'ai_x_px':a[0],'ai_y_px':a[1],
                      'edited_x_px':b[0],'edited_y_px':b[1],
                      'correction_distance_px':shift})
audit_df=pd.DataFrame(audit)
audit_file=out_dir/'hand_landmarks_manual_audit.csv'
audit_df.to_csv(audit_file,index=False)
corrected=source_df.copy()
corrected['annotation_status']='outside_edit_window'
corrected['xy_source']='original_media_pipe'
corrected['z_source']='original_media_pipe_unmodified'
for f in FRAME_IDS:
    for i in range(21):
        mask=(corrected.frame==f)&(corrected.landmark_id==i)
        p=edits[f][i];st=statuses[f][i]
        corrected.loc[mask,'annotation_status']=st
        corrected.loc[mask,'xy_source']=('manual_2d' if st=='manual_visible'
             else 'interpolated_2d' if st=='interpolated'
             else 'unobserved' if st in ('occluded','missing') else 'original_media_pipe')
        corrected.loc[mask,'x_px']=p[0]
        corrected.loc[mask,'y_px']=p[1]
        if 'x_norm' in corrected.columns:corrected.loc[mask,'x_norm']=p[0]/W
        if 'y_norm' in corrected.columns:corrected.loc[mask,'y_norm']=p[1]/H
        if st in ('occluded','missing'):
            corrected.loc[mask,'detected']=0
            corrected.loc[mask,'z_relative']=np.nan
            corrected.loc[mask,'z_source']='unobserved'
corrected_file=out_dir/'hand_21_landmarks_2d_corrected.csv'
corrected.to_csv(corrected_file,index=False)
print('Saved:',audit_file)
print('Saved:',corrected_file)
print('Statuses:',audit_df.status.value_counts().to_dict())
print('Manual shift mean (pixels):',
  audit_df.loc[audit_df.status=='manual_visible','correction_distance_px'].mean())